In [2]:
import pandas as pd
# import seaborn as sns
import numpy as np
# import matplotlib as mpl
# import matplotlib.pyplot as plt

In [7]:
df = pd.read_csv("/scratch/project/open-35-8/pimenol1/ProteinTTT/ProteinTTT_fresh/data/bfvd/experements_msa/best/results.tsv", sep="\t")
df

,id,nmsa,pLDDT_AlphaFold,version,sequence,length,sequence_length,pLDDT_ProteinTTT,lddt_ProteinTTT,tm_score_ProteinTTT,pLDDT_ESMFold,lddt_ESMFold,tm_score_ESMFold
0,A0A6J5N0Y1,335,91.472759,BASE,MEYKKQLLNELEKLTSLMDIPFARRTDIHWILQNVVMNNTDEKKIK...,62,62,86.124051,NaN,NaN,81.138626,NaN,NaN
1,A5A3S1,7,90.812526,BASE+LOGAN+12CY,MKEHVIIAPNYRLGEYAARFILEVSPRSCIFVSLDSRGDCERLNGL...,91,91,85.971750,NaN,NaN,63.134811,NaN,NaN
2,A0A646QXE5,55,91.541814,BASE+LOGAN,MQVKITANANALVLGTDTLIAANNELYTKLETCHVEVNPSTGKIVK...,89,89,60.141664,NaN,NaN,40.956676,NaN,NaN
3,A0A7T0Q4S0,203,92.460221,BASE+LOGAN+12CY,MYFPVTIKIIQIEDKLDITIEKGIEAEFVITRAICSCGFMFKIPII...,126,126,88.880490,NaN,NaN,33.140102,NaN,NaN
4,A0A8S5SDJ0,147,92.442504,BASE,MKMFAEVRNDYCFEELKNGIKAGITIDVWKTDDENEEGEVAATVIL...,82,82,82.380195,NaN,NaN,50.413876,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,S0A3B0,240,90.556291,BASE,MKKRFTKFKPQVYSRHPSHAPLRLKGALELLPFKSVVRFGSPTEVS...,289,289,89.851771,NaN,NaN,77.208038,NaN,NaN
96,A0A8S5N253,154,93.966714,BASE+LOGAN+12CY,MNAELAELIKEIGRIADTHEDNVEWYKIGDFCKIMPIFEEPESDAL...,94,94,80.772859,NaN,NaN,48.223984,NaN,NaN
97,A0A5P8PST7,2100,93.420123,BASE,MSLPNVKKPCKDCPFRKDTMKGWLGKERMQEILNQESFVCRKKTHL...,92,92,89.607763,NaN,NaN,50.468506,NaN,NaN
98,A0A6J7WB17,15,91.661256,BASE+LOGAN+12CY,MPRFAIFKGDGDLIFPVDAILVKYMKDEYLVRWVEADDASQAVENY...,50,50,87.164310,NaN,NaN,80.873413,NaN,NaN


In [11]:
df.pLDDT_AlphaFold.mean()

92.02101004365126

In [14]:
import sys
sys.path.insert(0, "/scratch/project/open-35-8/pimenol1/ProteinTTT/ProteinTTT_fresh")

from proteinttt.utils.structure import lddt_score
from pathlib import Path
from tqdm import tqdm

for dir in Path("/scratch/project/open-35-8/pimenol1/ProteinTTT/ProteinTTT_fresh/data/bfvd/experements_msa/experements_output").glob("*"):

    TRUE_PDB_DIR   = Path("/scratch/project/open-35-8/antonb/bfvd/bfvd")
    EXP_DIR        = dir
    PRED_TTT_DIR   = EXP_DIR / "predicted_structures" / "ESMFold_ProteinTTT"
    PRED_ESM_DIR   = EXP_DIR / "predicted_structures" / "ESMFold"

    lddt_ttt_list = []
    lddt_esm_list = []

    df = pd.read_csv(EXP_DIR / Path("results.tsv"), sep="\t")


    for _, row in tqdm(df.iterrows(), total=len(df)):
        pid = row["id"]
        true_pdb = TRUE_PDB_DIR / f"{pid}.pdb"
        pred_ttt = PRED_TTT_DIR / f"{pid}.pdb"
        pred_esm = PRED_ESM_DIR / f"{pid}.pdb"

        if true_pdb.exists() and pred_ttt.exists():
            try:
                score = lddt_score(str(true_pdb), str(pred_ttt))
            except Exception as e:
                print(f"[ProteinTTT] {pid}: {e}")
                score = float("nan")
        else:
            score = float("nan")
        lddt_ttt_list.append(score)

        # if true_pdb.exists() and pred_esm.exists():
        #     try:
        #         score = lddt_score(str(true_pdb), str(pred_esm))
        #     except Exception as e:
        #         print(f"[ESMFold] {pid}: {e}")
        #         score = float("nan")
        # else:
        #     score = float("nan")
        # lddt_esm_list.append(score)

    df["lddt_ProteinTTT"] = lddt_ttt_list
    # df["lddt_ESMFold"]    = lddt_esm_list
    print(dir)
    print(f"ProteinTTT  mean LDDT: {df['lddt_ProteinTTT'].mean():.4f}  (n={df['lddt_ProteinTTT'].notna().sum()})")
    # print(f"ESMFold     mean LDDT: {df['lddt_ESMFold'].mean():.4f}  (n={df['lddt_ESMFold'].notna().sum()})")
    # df[["id", "lddt_ProteinTTT", "lddt_ESMFold"]]

100%|██████████| 100/100 [00:03<00:00, 30.60it/s]


/scratch/project/open-35-8/pimenol1/ProteinTTT/ProteinTTT_fresh/data/bfvd/experements_msa/experements_output/lr_0.04_ags_8_msa_True_grad_50.0_random_3962093
ProteinTTT  mean LDDT: 0.3633  (n=100)


100%|██████████| 100/100 [00:03<00:00, 29.54it/s]


/scratch/project/open-35-8/pimenol1/ProteinTTT/ProteinTTT_fresh/data/bfvd/experements_msa/experements_output/lr_0.04_ags_8_msa_True_grad_1.0_top_3959419
ProteinTTT  mean LDDT: 0.6827  (n=100)


100%|██████████| 100/100 [00:03<00:00, 30.05it/s]


...

/scratch/project/open-35-8/pimenol1/ProteinTTT/ProteinTTT_fresh/data/bfvd/experements_msa/experements_output/lr_0.04_ags_32_msa_True_grad_1.0_random_3961323
ProteinTTT  mean LDDT: 0.8653  (n=100)


100%|██████████| 100/100 [00:03<00:00, 29.94it/s]


/scratch/project/open-35-8/pimenol1/ProteinTTT/ProteinTTT_fresh/data/bfvd/experements_msa/experements_output/lr_0.04_ags_8_msa_True_grad_1.0_cluster_3963070
ProteinTTT  mean LDDT: 0.7756  (n=100)


100%|██████████| 100/100 [00:03<00:00, 29.96it/s]

/scratch/project/open-35-8/pimenol1/ProteinTTT/ProteinTTT_fresh/data/bfvd/experements_msa/experements_output/lr_0.04_ags_8_msa_True_grad_1.0_random_3962387
ProteinTTT  mean LDDT: 0.8316  (n=100)
